# Exercice 4 : Représentation et Description d'Images
## TP1 — Vision par Ordinateur

---

**Objectifs :**
1. Charger une image couleur
2. Convertir en niveaux de gris
3. Détecter les contours
4. Calculer l'histogramme
5. Détecter des points clés locaux (ORB)
6. Appliquer la Transformée de Fourier
7. Appliquer un filtre de Gabor
8. Sauvegarder les résultats

---

## Étape 1 — Installation

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

print("Prêt !")

## Étape 2 — Charger une image couleur

In [ ]:
# Télécharger une image
import urllib.request

url = "https://ultralytics.com/images/bus.jpg"
urllib.request.urlretrieve(url, "image.jpg")

# Lire l'image couleur
img = cv2.imread("image.jpg")

# Convertir BGR -> RGB pour l'affichage avec matplotlib
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.axis('off')
plt.title('Image couleur originale')
plt.show()

## Étape 3 — Convertir en niveaux de gris

In [ ]:
# Convertir en gris
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(8, 6))
plt.imshow(gray, cmap='gray')
plt.axis('off')
plt.title('Niveaux de gris')
plt.show()

---
## Étape 4 — Détection de contours (Canny)

L'algorithme de Canny détecte les bords des objets en trouvant les changements brutaux d'intensité.

In [ ]:
# Détecter les contours avec Canny
edges = cv2.Canny(gray, 100, 200)

plt.figure(figsize=(10, 7))
plt.imshow(edges, cmap='gray')
plt.axis('off')
plt.title('Contours détectés (Canny 100, 200)')
plt.show()

## Étape 5 — Histogramme

L'histogramme montre la répartition des valeurs de pixels (0 = sombre, 255 = clair).

In [ ]:
# Afficher l'histogramme
plt.figure(figsize=(10, 5))
plt.hist(gray.ravel(), 256, [0, 256])
plt.title('Histogramme des niveaux de gris', fontsize=14)
plt.xlabel('Valeur de pixel')
plt.ylabel('Nombre de pixels')
plt.grid(True, alpha=0.3)
plt.show()

---
## Étape 6 — Détection de points clés locaux (ORB)

**ORB** (Oriented FAST and Rotated BRIEF) est un algorithme qui trouve des **points d'intérêt** dans une image.

**C'est quoi un point clé ?**
Un endroit particulier de l'image : un coin, une intersection, un détail marquant.

**À quoi ça sert ?**
- Comparer deux images entre elles
- Suivre un objet dans une vidéo
- Reconnaître des objets

In [ ]:
# Créer l'objet ORB
orb = cv2.ORB_create()

# Détecter les points clés et calculer les descripteurs
keypoints, descriptors = orb.detectAndCompute(gray, None)

print(f"Nombre de points clés trouvés : {len(keypoints)}")
if descriptors is not None:
    print(f"Taille du descripteur : {descriptors.shape}")

In [ ]:
# Dessiner les points clés sur l'image
img_orb = cv2.drawKeypoints(img_rgb, keypoints, None, color=(0, 255, 0), flags=0)

plt.figure(figsize=(12, 8))
plt.imshow(img_orb)
plt.axis('off')
plt.title(f'Points clés ORB détectés ({len(keypoints)} points)')
plt.show()

---
## Étape 7 — Transformée de Fourier

La **Transformée de Fourier** transforme une image de l'espace spatial (pixels) vers l'espace fréquentiel (fréquences).

**Pourquoi ?**
- Les basses fréquences = zones homogènes (fond, ciel)
- Les hautes fréquences = détails, contours, textures

**Utilité :** On peut filtrer certaines fréquences (supprimer le bruit, accentuer les contours...)

In [ ]:
# Calculer la Transformée de Fourier
f = np.fft.fft2(gray)

# Déplacer la composante DC (fréquence 0) au centre
f_shift = np.fft.fftshift(f)

# Calculer le spectre en magnitude (en log pour bien voir)
magnitude = 20 * np.log(np.abs(f_shift) + 1)

# Afficher
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Image originale', fontsize=14)
axes[0].axis('off')

axes[1].imshow(magnitude, cmap='gray')
axes[1].set_title('Spectre de Fourier', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("Le point brillant au centre = la fréquence 0 (composante DC)")
print("Les points proches du centre = basses frézones (zones lisses)")
print("Les points éloignés du centre = hautes fréquences (détails, contours)")

### Filtrage passe-bas (supprimer les hautes fréquences)

On masque les hautes fréquences pour garder seulement les basses = on floute l'image.

In [ ]:
# Créer un masque circulaire (passe-bas)
rows, cols = gray.shape
center_row, center_col = rows // 2, cols // 2

# Rayon du cercle : on garde que les basses fréquences
rayon = 30

# Masque : 1 à l'intérieur du cercle, 0 à l'extérieur
masque = np.zeros((rows, cols), np.uint8)
masque[center_row - rayon:center_row + rayon, center_col - rayon:center_col + rayon] = 1

# Appliquer le masque
f_shift_filtre = f_shift * masque

# Transformée inverse
f_ishift = np.fft.ifftshift(f_shift_filtre)
img_filtree = np.fft.ifft2(f_ishift)
img_filtree = np.abs(img_filtree)

# Afficher
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Original', fontsize=14)
axes[0].axis('off')

axes[1].imshow(masque, cmap='gray')
axes[1].set_title(f'Masque passe-bas (rayon={rayon})', fontsize=14)
axes[1].axis('off')

axes[2].imshow(img_filtree, cmap='gray')
axes[2].set_title('Image filtrée (flou)', fontsize=14)
axes[2].axis('off')

plt.tight_layout()
plt.show()

---
## Étape 8 — Filtre de Gabor

Le filtre de Gabor détecte des **textures** dans une direction précise.

**C'est quoi ?** C'est un filtre qui ressemble à une onde. Il est sensible à :
- La **direction** de la texture
- La **fréquence** de la texture

**Utilité :** Détection de bordures, textures dans les images médicales, reconnaissance de textures...

In [ ]:
# Créer un filtre de Gabor
# theta = angle du filtre en radians
theta = 0  # angle horizontal

kernel_size = 21   # taille du filtre
sigma = 5          # écart-type du gaussien
lambd = 10         # longueur d'onde
gamma = 0.5        # rapport d'aspect

gabor_kernel = cv2.getGaborKernel(
    (kernel_size, kernel_size),
    sigma,
    theta,
    lambd,
    gamma
)

# Appliquer le filtre sur l'image en gris
gabor_result = cv2.filter2D(gray, cv2.CV_8UC3, gabor_kernel)

# Afficher le filtre et le résultat
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Original', fontsize=14)
axes[0].axis('off')

axes[1].imshow(gabor_kernel, cmap='gray')
axes[1].set_title('Noyau de Gabor', fontsize=14)
axes[1].axis('off')

axes[2].imshow(gabor_result, cmap='gray')
axes[2].set_title('Résultat Gabor (θ=0°)', fontsize=14)
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Tester plusieurs angles

Le filtre de Gabor réagit différemment selon l'angle. Essayons plusieurs directions.

In [ ]:
# Tester 4 angles différents
angles = [0, 45, 90, 135]  # en degrés

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, angle_deg in enumerate(angles):
    # Convertir en radians
    angle_rad = angle_deg * np.pi / 180

    # Créer le noyau avec cet angle
    kernel = cv2.getGaborKernel(
        (21, 21),  # taille
        5,         # sigma
        angle_rad, # angle
        10,        # lambda
        0.5        # gamma
    )

    # Appliquer
    result = cv2.filter2D(gray, cv2.CV_8UC3, kernel)

    axes[i].imshow(result, cmap='gray')
    axes[i].set_title(f'θ = {angle_deg}°', fontsize=14)
    axes[i].axis('off')

plt.suptitle('Filtre de Gabor - Différents angles', fontsize=16)
plt.tight_layout()
plt.show()

---
## Étape 9 — Sauvegarder les résultats

In [ ]:
# Sauvegarder les images générées
cv2.imwrite('contours.png', edges)
cv2.imwrite('histogramme.png', gray)  # on sauvegarde le gris pour référence

# Sauvegarder l'ORB
cv2.imwrite('orb_keypoints.png', cv2.cvtColor(img_orb, cv2.COLOR_RGB2BGR))

# Sauvegarder Fourier
cv2.imwrite('fourier_spectre.png', magnitude.astype(np.uint8))

# Sauvegarder Gabor
cv2.imwrite('gabor_result.png', gabor_result)

print("Toutes les images ont été sauvegardées !")

---
## Résumé

| Technique | Rôle | Code |
|-----------|------|------|
| **Canny** | Détecter les contours | `cv2.Canny(gray, 100, 200)` |
| **Histogramme** | Distribution des pixels | `plt.hist(gray.ravel(), 256, [0,256])` |
| **ORB** | Points d'intérêt | `cv2.ORB_create()` + `orb.detectAndCompute()` |
| **Fourier** | Analyse fréquentielle | `np.fft.fft2(gray)` |
| **Gabor** | Détection de textures | `cv2.getGaborKernel()` + `cv2.filter2D()` |

**À retenir :**
- ORB trouve des points intéressants (coins, textures)
- Fourier transforme l'image en fréquences
- Gabor détecte les textures dans une direction précise